# Audit evidence: Iris control

## Question and record scope

The request is to summarize the four measurements by species in this sample, compare them descriptively, and state limits. The extract has 150 rows, 50 per species, and measurements are in centimeters. It is a sample summary, not a documented probability sample of all iris populations.

## Reproduction and data checks

The saved submitted-code run exited successfully and printed its aggregate summaries and boxplot confirmation. Independent computations reproduce all 12 species-by-measure means and 95% t intervals. Bootstrap percentile intervals for the means are close to the t intervals. The source has no missing values or nonpositive measurements and contains three exact duplicate rows; they are retained because identical rounded measurements alone do not prove repeated specimens.

## Interpretation of comparisons

All pairwise mean differences are reported with pointwise 95% bootstrap intervals. There are 12 contrasts across four measurements; the intervals are not familywise/simultaneous intervals. The response explicitly treats these as descriptive and avoids p-value hunting, so no confirmatory interpretation is attributed to them.

Observed petal ranges for setosa do not overlap those of either other species. Versicolor and virginica petal ranges overlap. This supports descriptive separation in this sample, not validated classification accuracy.

## Limits

The t and bootstrap intervals concern species means under independent-row assumptions. They are not prediction intervals for individual flowers. Unknown sampling provenance limits population generalization; the summaries do not explain biological causes.

Run the code cell to regenerate aggregate evidence. It does not print source rows.

In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'audit_tools.py').is_file())
sys.path.insert(0, str(project_root / 'src'))
from iris_audit import audit_iris_control

audit = audit_iris_control()
print({
    'n': audit['n_total'],
    'species_counts': audit['species_counts'],
    'missing_values_by_column': audit['missing_values_by_column'],
    'exact_duplicate_rows': audit['duplicate_rows'],
    'nonpositive_measurement_rows': audit['nonpositive_measurement_rows'],
    'saved_code_exit_code': audit['saved_ai_run']['exit_code'],
    'reference_max_mean_difference_cm': audit['reference_comparison']['max_absolute_mean_difference_cm'],
    'reference_max_t_ci_endpoint_difference_cm': audit['reference_comparison']['max_absolute_t_interval_endpoint_difference_cm'],
})

summary = pd.DataFrame(audit['summary_statistics'])
summary['t 95% CI (cm)'] = summary['t_ci_95_cm'].map(lambda ci: f'{ci[0]:.3f} to {ci[1]:.3f}')
summary['bootstrap 95% CI (cm)'] = summary['bootstrap_ci_95_cm'].map(lambda ci: f'{ci[0]:.3f} to {ci[1]:.3f}')
display(summary[['species', 'measurement', 'n', 'mean_cm', 'sd_cm', 'median_cm', 'min_cm', 'max_cm', 't 95% CI (cm)', 'bootstrap 95% CI (cm)']].round(3))

contrasts = pd.DataFrame(audit['pairwise_comparisons'])
contrasts['contrast'] = contrasts['first_species'] + ' - ' + contrasts['second_species']
contrasts['bootstrap 95% CI (cm)'] = contrasts['bootstrap_ci_95_cm'].map(lambda ci: f'{ci[0]:+.3f} to {ci[1]:+.3f}')
display(contrasts[['measurement', 'contrast', 'mean_difference_cm', 'bootstrap 95% CI (cm)', 'cohens_d']].round(3))

display(pd.DataFrame(audit['observed_range_overlaps']))

{'n': 150, 'species_counts': {'Iris-setosa': 50, 'Iris-versicolor': 50, 'Iris-virginica': 50}, 'missing_values_by_column': {'sepal length': 0, 'sepal width': 0, 'petal length': 0, 'petal width': 0, 'class': 0}, 'exact_duplicate_rows': 3, 'nonpositive_measurement_rows': 0, 'saved_code_exit_code': 0, 'reference_max_mean_difference_cm': 0.0, 'reference_max_t_ci_endpoint_difference_cm': 0.0}


,species,measurement,n,mean_cm,sd_cm,median_cm,min_cm,max_cm,t 95% CI (cm),bootstrap 95% CI (cm)
0,Iris-setosa,sepal length,50,5.006,0.352,5.00,4.3,5.8,4.906 to 5.106,4.908 to 5.100
1,Iris-setosa,sepal width,50,3.418,0.381,3.40,2.3,4.4,3.310 to 3.526,3.314 to 3.522
2,Iris-setosa,petal length,50,1.464,0.174,1.50,1.0,1.9,1.415 to 1.513,1.416 to 1.512
3,Iris-setosa,petal width,50,0.244,0.107,0.20,0.1,0.6,0.214 to 0.274,0.216 to 0.274
4,Iris-versicolor,sepal length,50,5.936,0.516,5.90,4.9,7.0,5.789 to 6.083,5.798 to 6.078
5,Iris-versicolor,sepal width,50,2.770,0.314,2.80,2.0,3.4,2.681 to 2.859,2.680 to 2.854
6,Iris-versicolor,petal length,50,4.260,0.470,4.35,3.0,5.1,4.126 to 4.394,4.124 to 4.386
7,Iris-versicolor,petal width,50,1.326,0.198,1.30,1.0,1.8,1.270 to 1.382,1.274 to 1.382
8,Iris-virginica,sepal length,50,6.588,0.636,6.50,4.9,7.9,6.407 to 6.769,6.416 to 6.766
9,Iris-virginica,sepal width,50,2.974,0.322,3.00,2.2,3.8,2.882 to 3.066,2.886 to 3.064


,measurement,contrast,mean_difference_cm,bootstrap 95% CI (cm),cohens_d
0,sepal length,Iris-setosa - Iris-versicolor,-0.930,-1.098 to -0.758,-2.104
1,sepal length,Iris-setosa - Iris-virginica,-1.582,-1.788 to -1.382,-3.077
2,sepal length,Iris-versicolor - Iris-virginica,-0.652,-0.876 to -0.428,-1.126
3,sepal width,Iris-setosa - Iris-versicolor,0.648,+0.516 to +0.780,1.857
4,sepal width,Iris-setosa - Iris-virginica,0.444,+0.306 to +0.578,1.258
5,sepal width,Iris-versicolor - Iris-virginica,-0.204,-0.328 to -0.078,-0.641
6,petal length,Iris-setosa - Iris-versicolor,-2.796,-2.930 to -2.654,-7.894
7,petal length,Iris-setosa - Iris-virginica,-4.088,-4.248 to -3.936,-9.993
8,petal length,Iris-versicolor - Iris-virginica,-1.292,-1.490 to -1.096,-2.521
9,petal width,Iris-setosa - Iris-versicolor,-1.082,-1.144 to -1.020,-6.802


,measurement,first_species,second_species,first_range_cm,second_range_cm,observed_ranges_overlap,overlap_width_cm
0,sepal length,Iris-setosa,Iris-versicolor,"[4.3, 5.8]","[4.9, 7.0]",True,0.9
1,sepal length,Iris-setosa,Iris-virginica,"[4.3, 5.8]","[4.9, 7.9]",True,0.9
2,sepal length,Iris-versicolor,Iris-virginica,"[4.9, 7.0]","[4.9, 7.9]",True,2.1
3,sepal width,Iris-setosa,Iris-versicolor,"[2.3, 4.4]","[2.0, 3.4]",True,1.1
4,sepal width,Iris-setosa,Iris-virginica,"[2.3, 4.4]","[2.2, 3.8]",True,1.5
5,sepal width,Iris-versicolor,Iris-virginica,"[2.0, 3.4]","[2.2, 3.8]",True,1.2
6,petal length,Iris-setosa,Iris-versicolor,"[1.0, 1.9]","[3.0, 5.1]",False,0.0
7,petal length,Iris-setosa,Iris-virginica,"[1.0, 1.9]","[4.5, 6.9]",False,0.0
8,petal length,Iris-versicolor,Iris-virginica,"[3.0, 5.1]","[4.5, 6.9]",True,0.6
9,petal width,Iris-setosa,Iris-versicolor,"[0.1, 0.6]","[1.0, 1.8]",False,0.0
